# <font color="#418FDE" size="6.5" uppercase>**Neighborhood Manifolds**</font>

>Last update: 20260714.
    
By the end of this Lecture, you will be able to:
- Explain the manifold assumption through visual examples. 
- Apply Isomap and LLE-family methods to nonlinear datasets. 
- Analyze neighborhood construction, solver choices, and out-of-sample transformation behavior. 


## **1. Manifold Intuition**

### **1.1. Manifold Assumption**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_A/image_01_01.jpg?v=1784027685" width="250">



>* High-dimensional data may hide simpler surfaces
>* Few factors can explain many measurements

>* Data often follows hidden curved shapes
>* Simple factors can drive high-dimensional measurements

>* Measure closeness along the data surface
>* Noise can hide useful manifold structure



### **1.2. Isomap Intuition**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_A/image_01_02.jpg?v=1784027684" width="250">



>* Preserve distances along the curved data surface
>* Unfold hidden structure using surface paths

>* Swiss roll shows misleading straight-line distances
>* Isomap follows neighborhood paths along the surface

>* High-dimensional data can have simple hidden factors
>* Isomap preserves meaningful paths on the manifold



In [ ]:
#@title Python Code - Isomap Intuition

# This example visualizes Isomap on a Swiss roll.
# Isomap preserves walking distances along curved data.
# The plot shows an unfolded two-dimensional map.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import Isomap

# Generate a small curved surface in three dimensions.
points_3d, roll_position = make_swiss_roll(
    n_samples=900,
    noise=0.05,
    random_state=42,
)

# Validate the generated data before fitting Isomap.
if points_3d.shape != (900, 3):
    raise ValueError("Expected 900 points with three coordinates.")

# Fit Isomap to estimate distances along the roll.
isomap = Isomap(n_neighbors=10, n_components=2)
points_2d = isomap.fit_transform(points_3d)

# Print concise context for the visual result.
print(f"scikit-learn version: {sklearn.__version__}")
print("Swiss roll points: 900 in 3D, mapped to 2D.")
print("Color follows position along the original rolled sheet.")

# Plot the unfolded Isomap coordinates as one clear map.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    points_2d[:, 0],
    points_2d[:, 1],
    c=roll_position,
    cmap="viridis",
    s=18,
)

# Label the map to emphasize intrinsic geometry.
ax.set_title("Isomap unfolds a Swiss roll manifold")
ax.set_xlabel("Isomap coordinate 1")
ax.set_ylabel("Isomap coordinate 2")
fig.colorbar(scatter, ax=ax, label="Position along rolled sheet")

plt.show()



### **1.3. Geodesic Neighborhoods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_A/image_01_03.jpg?v=1784027687" width="250">



>* Manifold distance follows the curved structure
>* Straight-line closeness can create false neighbors

>* Surface distance can differ from physical closeness
>* Neighbors should reflect smooth manifold movement

>* Connect local neighbors to trace manifold paths
>* Good neighborhoods preserve geometry, avoiding false links



In [ ]:
#@title Python Code - Geodesic Neighborhoods

# This script compares straight and manifold distances.
# A crescent shape reveals misleading Euclidean shortcuts.
# The plot highlights geodesic neighborhood intuition.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import pairwise_distances

# Generate a small curved dataset with deterministic noise.
points, labels = make_moons(n_samples=160, noise=0.04, random_state=42)
if points.shape != (160, 2):
    raise ValueError("Expected 160 two-dimensional points.")

# Choose two points on opposite ends of the same crescent.
crescent = np.where(labels == 0)[0]
start_index = crescent[np.argmin(points[crescent, 0])]
end_index = crescent[np.argmax(points[crescent, 0])]

# Build local roads using only nearby neighbors.
graph = kneighbors_graph(points, n_neighbors=6, mode="distance")
graph = graph.maximum(graph.T).tocsr()
distance_matrix = pairwise_distances(points)

# Compute shortest paths through the neighborhood graph.
n = len(points)
visited = np.zeros(n, dtype=bool)
geodesic = np.full(n, np.inf)
previous = np.full(n, -1, dtype=int)
geodesic[start_index] = 0.0

for step in range(n):
    candidates = np.where(~visited, geodesic, np.inf)
    current = int(np.argmin(candidates))
    if not np.isfinite(geodesic[current]):
        break
    visited[current] = True
    neighbors = graph[current].nonzero()[1]
    for neighbor in neighbors:
        new_distance = geodesic[current] + distance_matrix[current, neighbor]
        if new_distance < geodesic[neighbor]:
            geodesic[neighbor] = new_distance
            previous[neighbor] = current

# Reconstruct the local-road path between the two points.
path = [end_index]
while path[-1] != start_index:
    next_index = previous[path[-1]]
    if next_index == -1:
        raise ValueError("Neighborhood graph is disconnected.")
    path.append(int(next_index))
path = path[::-1]

# Print the key distance comparison.
straight_distance = distance_matrix[start_index, end_index]
print("Euclidean shortcut distance:", round(float(straight_distance), 2))
print("Neighborhood graph distance:", round(float(geodesic[end_index]), 2))
print("Path uses local steps:", len(path) - 1)

# Draw one figure showing shortcut versus geodesic path.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(points[:, 0], points[:, 1], c=labels, cmap="coolwarm", s=35)
ax.plot(points[[start_index, end_index], 0], points[[start_index, end_index], 1],
        color="black", linestyle="--", linewidth=2, label="Euclidean shortcut")
ax.plot(points[path, 0], points[path, 1], color="gold", linewidth=3,
        label="Geodesic neighborhood path")
ax.scatter(points[[start_index, end_index], 0], points[[start_index, end_index], 1],
           color="lime", edgecolor="black", s=120, label="Chosen points")

ax.set_title("Geodesic Neighborhoods Avoid Shortcuts")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(loc="best")
plt.show()



## **2. LLE Family**

### **2.1. Local Linear Structure**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_A/image_02_01.jpg?v=1784027691" width="250">



>* Small neighborhoods act like flat patches
>* Neighbor relationships reveal lower-dimensional structure

>* Reconstruct each point from nearby neighbors
>* Preserve local weights in lower dimensions

>* Choose neighborhoods that reflect local geometry
>* Dense sampling helps reveal intrinsic structure



In [ ]:
#@title Python Code - Local Linear Structure

# Demonstrate local linear structure in LLE.
# Compare neighborhoods on a curved manifold.
# Show how LLE unfolds local relationships.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_s_curve
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.neighbors import NearestNeighbors

# Create a small curved dataset with known color order.
points_3d, color_value = make_s_curve(n_samples=700, noise=0.04, random_state=42)

# Validate the generated data before fitting models.
if points_3d.shape != (700, 3):
    raise ValueError("Expected 700 samples with three coordinates.")

# Fit LLE using small neighborhoods that are nearly flat.
neighbors = 12
lle = LocallyLinearEmbedding(
    n_neighbors=neighbors,
    n_components=2,
    method="standard",
    eigen_solver="auto",
)

embedding = lle.fit_transform(points_3d)

# Measure how well each point is reconstructed locally.
nearest = NearestNeighbors(n_neighbors=neighbors + 1)
nearest.fit(points_3d)
distances, indices = nearest.kneighbors(points_3d)

# Ignore each point itself when summarizing neighborhood size.
mean_neighbor_distance = distances[:, 1:].mean()
reconstruction_error = lle.reconstruction_error_

print("scikit-learn version:", sklearn.__version__)
print("Dataset: synthetic S-curve with 700 points.")
print("LLE neighbors:", neighbors)
print("Mean neighbor distance:", round(float(mean_neighbor_distance), 3))
print("LLE reconstruction error:", round(float(reconstruction_error), 6))

# Plot the two LLE coordinates colored by manifold position.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=color_value,
    cmap="viridis",
    s=18,
)

ax.set_title("LLE preserves local linear neighborhoods")
ax.set_xlabel("LLE coordinate 1")
ax.set_ylabel("LLE coordinate 2")
fig.colorbar(scatter, ax=ax, label="Position along S-curve")
plt.show()



### **2.2. Modified LLE**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_A/image_02_02.jpg?v=1784027689" width="250">



>* Stabilizes LLE for imperfect local neighborhoods
>* Preserves manifolds despite noise and sparse sampling

>* Captures multiple local directions in neighborhoods
>* Produces smoother, more reliable low-dimensional maps

>* Choose neighborhood size carefully for local linearity
>* Reveal robust nonlinear structure beyond compression



In [ ]:
#@title Python Code - Modified LLE

# This example compares original and modified LLE.
# A noisy Swiss roll shows nonlinear neighborhood structure.
# The plot reveals how embeddings preserve local geometry.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_swiss_roll
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.preprocessing import StandardScaler

# Generate a small curved dataset with deterministic noise.
points, roll_position = make_swiss_roll(
    n_samples=700,
    noise=0.08,
    random_state=42,
)

# Scaling keeps distance-based neighborhoods balanced across coordinates.
scaled_points = StandardScaler().fit_transform(points)

# Validate the shape before fitting neighborhood embeddings.
if scaled_points.shape != (700, 3):
    raise ValueError("Expected 700 samples with 3 coordinates.")

# Modified LLE uses richer local information than standard LLE.
modified_lle = LocallyLinearEmbedding(
    n_neighbors=18,
    n_components=2,
    method="modified",
    random_state=42,
)

# Fit the manifold model and compute two embedding coordinates.
embedding = modified_lle.fit_transform(scaled_points)

# Print concise context for the teaching example.
print("scikit-learn version:", sklearn.__version__)
print("Dataset: noisy Swiss roll with 700 samples.")
print("Method: Modified LLE with 18 nearest neighbors.")

# Color points by their true position along the rolled manifold.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=roll_position,
    cmap="viridis",
    s=18,
)

# Labels describe the learned low-dimensional coordinates.
ax.set_title("Modified LLE embedding of a noisy Swiss roll")
ax.set_xlabel("Modified LLE coordinate 1")
ax.set_ylabel("Modified LLE coordinate 2")
fig.colorbar(scatter, ax=ax, label="Position along original roll")

plt.show()



### **2.3. Hessian Eigenmapping**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_A/image_02_03.jpg?v=1784027693" width="250">



>* Preserves smooth manifold geometry using curvature
>* Finds low-dimensional coordinates with minimal bending

>* Estimate tangent directions within local neighborhoods
>* Unfold smooth hidden motion coordinates

>* Needs well-chosen, reliable local neighborhoods
>* Smooth, dense data reveals manifold coordinates



In [ ]:
#@title Python Code - Hessian Eigenmapping

# This script demonstrates Hessian Eigenmapping on curved data.
# It compares original height with learned coordinates.
# The plot should reveal a smooth manifold parameter.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_s_curve
from sklearn.manifold import LocallyLinearEmbedding

# Generate a small nonlinear manifold with known position values.
points, position = make_s_curve(n_samples=700, noise=0.04, random_state=42)

# Validate the generated data before fitting the embedding.
if points.shape != (700, 3):
    raise ValueError("Expected 700 samples with three measured features.")

# Hessian LLE needs enough neighbors for curvature estimation.
embedding_model = LocallyLinearEmbedding(
    n_neighbors=18,
    n_components=2,
    method="hessian",
    eigen_solver="auto"
)

# Fit the manifold model and transform the data to two coordinates.
embedding = embedding_model.fit_transform(points)

# Print concise context for the teaching example.
print("scikit-learn version:", sklearn.__version__)
print("Input shape:", points.shape)
print("Hessian embedding shape:", embedding.shape)

# Color shows the hidden position along the original S curve.
figure, axis = plt.subplots(figsize=(7, 5))
scatter = axis.scatter(
    embedding[:, 0], embedding[:, 1], c=position, cmap="viridis", s=18
)

# Label the learned coordinates for beginner interpretation.
axis.set_title("Hessian Eigenmapping of an S-curve manifold")
axis.set_xlabel("Learned coordinate 1")
axis.set_ylabel("Learned coordinate 2")

# The colorbar links the embedding to the original manifold position.
colorbar = figure.colorbar(scatter, ax=axis)
colorbar.set_label("Position along original S curve")
plt.show()



## **3. Manifold Practice**

### **3.1. Local Tangent Alignment**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_A/image_03_01.jpg?v=1784027678" width="250">



>* Small neighborhoods approximate curved manifolds locally
>* Aligned local coordinates reveal lower-dimensional structure

>* Neighborhood choices shape tangent estimates
>* Avoid noise, shortcuts, and misleading neighbors

>* Align local patches into global coordinates
>* Validate embeddings against sampling and artifacts



In [ ]:
#@title Python Code - Local Tangent Alignment

# This example visualizes local tangent alignment.
# Neighborhood size changes each tangent estimate.
# The plot reveals stable and misleading neighborhoods.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_s_curve
from sklearn.neighbors import NearestNeighbors

# Generate a small curved manifold in three dimensions.
points, color_value = make_s_curve(n_samples=450, noise=0.04, random_state=42)
points = points[:, [0, 2]]

# Validate the two-dimensional teaching view.
if points.shape != (450, 2):
    raise ValueError("Expected 450 points with two plotted coordinates.")

# Choose one point where local neighborhoods are compared.
anchor_index = 225
anchor_point = points[anchor_index]

# Estimate tangent directions from small and large neighborhoods.
neighborhood_sizes = [8, 55]
tangent_segments = []

for size in neighborhood_sizes:
    neighbors = NearestNeighbors(n_neighbors=size)
    neighbors.fit(points)
    neighbor_ids = neighbors.kneighbors([anchor_point], return_distance=False)[0]

    local_points = points[neighbor_ids]
    centered_points = local_points - local_points.mean(axis=0)
    _, _, directions = np.linalg.svd(centered_points, full_matrices=False)

    tangent_direction = directions[0]
    segment_length = 1.2
    start = anchor_point - segment_length * tangent_direction
    end = anchor_point + segment_length * tangent_direction
    tangent_segments.append((size, start, end))

# Print concise teaching notes about the neighborhood choice.
print("scikit-learn version: imported through sklearn APIs")
print("Small neighborhood: follows a local flat patch.")
print("Large neighborhood: can cut across the curved manifold.")

# Plot the manifold and the two estimated tangent directions.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(points[:, 0], points[:, 1], c=color_value, s=18, cmap="viridis")
ax.scatter(anchor_point[0], anchor_point[1], color="red", s=80, label="anchor point")

for size, start, end in tangent_segments:
    ax.plot([start[0], end[0]], [start[1], end[1]], linewidth=3, label=f"k={size} tangent")

ax.set_title("Local tangent estimates depend on neighborhood size")
ax.set_xlabel("S-curve coordinate 1")
ax.set_ylabel("S-curve coordinate 2")
ax.legend()
plt.show()



### **3.2. Eigen Solver Choices**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_A/image_03_02.jpg?v=1784027680" width="250">



>* Embeddings depend on eigen solver choices
>* Sparse solvers scale better for large data

>* Match solvers to data size and precision
>* Close eigenvalues can change embedding directions

>* Judge solvers by embedding stability
>* Test settings to avoid numerical artifacts



In [ ]:
#@title Python Code - Eigen Solver Choices

# Compare eigen solvers for one manifold embedding.
# Solver choice can change speed and stability.
# Similar embeddings suggest a reliable numerical result.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_s_curve
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.metrics import pairwise_distances

# A small curved dataset keeps the spectral problem beginner friendly.
points, color_value = make_s_curve(n_samples=450, noise=0.04, random_state=42)

# This check catches unexpected data-shape problems early.
if points.shape != (450, 3):
    raise ValueError("Expected 450 samples with 3 features.")

# Two solver settings estimate the same two-dimensional manifold coordinates.
dense_lle = LocallyLinearEmbedding(
    n_neighbors=12, n_components=2, eigen_solver="dense", random_state=42
)

arpack_lle = LocallyLinearEmbedding(
    n_neighbors=12, n_components=2, eigen_solver="arpack", random_state=42
)

# Fit each solver once, then compare neighborhood distances in the embeddings.
dense_embedding = dense_lle.fit_transform(points)
arpack_embedding = arpack_lle.fit_transform(points)

# Distances are centered and scaled before comparing solver outcomes.
dense_distances = pairwise_distances(dense_embedding)
arpack_distances = pairwise_distances(arpack_embedding)

# Correlation near one means the solvers preserved similar relationships.
dense_flat = dense_distances.ravel()
arpack_flat = arpack_distances.ravel()
correlation = np.corrcoef(dense_flat, arpack_flat)[0, 1]

print("scikit-learn version:", sklearn.__version__)
print("Dense reconstruction error:", round(float(dense_lle.reconstruction_error_), 6))
print("ARPACK reconstruction error:", round(float(arpack_lle.reconstruction_error_), 6))
print("Embedding distance correlation:", round(float(correlation), 4))

# The plot shows one solver result; printed values compare both solvers.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    dense_embedding[:, 0], dense_embedding[:, 1], c=color_value, s=18
)

ax.set_title("LLE embedding using the dense eigen solver")
ax.set_xlabel("Embedding coordinate 1")
ax.set_ylabel("Embedding coordinate 2")
plt.show()



### **3.3. Transforming New Data**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_25/Lecture_A/image_03_03.jpg?v=1784027682" width="250">



>* New points need neighborhood-based placement
>* Reliable transforms require matching manifold structure

>* Use neighbors to place new points
>* Assume local structure matches the manifold

>* Test new-point behavior for hidden weaknesses
>* Check neighborhoods, metrics, and out-of-distribution cases



In [ ]:
#@title Python Code - Transforming New Data

# Demonstrate transforming new manifold data.
# Compare familiar and unusual new points.
# Show neighbor distance as a warning.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_s_curve
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.neighbors import NearestNeighbors

# Create a small curved training manifold.
training_data, training_color = make_s_curve(
    n_samples=500,
    noise=0.04,
    random_state=42,
)

# Add new points, including one off the learned manifold.
new_data = np.array([
    training_data[10],
    training_data[120],
    [0.0, 3.0, 0.0],
])
new_labels = ["known-like A", "known-like B", "off-manifold"]

# Validate the feature shapes before fitting.
if training_data.shape[1] != new_data.shape[1]:
    raise ValueError("Training and new data must have matching features.")

# Fit one neighborhood manifold model on training data only.
lle = LocallyLinearEmbedding(
    n_neighbors=12,
    n_components=2,
    method="standard",
    eigen_solver="auto",
)
training_embedding = lle.fit_transform(training_data)

# Transform new points through their training neighborhoods.
new_embedding = lle.transform(new_data)

# Measure how close each new point is to the training manifold.
neighbors = NearestNeighbors(n_neighbors=12)
neighbors.fit(training_data)
distances, indices = neighbors.kneighbors(new_data)
mean_neighbor_distance = distances.mean(axis=1)

print("scikit-learn version:", sklearn.__version__)
print("New point        mean neighbor distance")
for label, distance in zip(new_labels, mean_neighbor_distance):
    print(label.ljust(16), round(float(distance), 3))

# Plot training embedding and transformed new points together.
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    training_embedding[:, 0],
    training_embedding[:, 1],
    c=training_color,
    s=18,
    cmap="viridis",
    alpha=0.65,
)

ax.scatter(
    new_embedding[:, 0],
    new_embedding[:, 1],
    c=["red", "orange", "black"],
    s=90,
    marker="X",
    label="transformed new points",
)

for label, x_value, y_value in zip(
    new_labels,
    new_embedding[:, 0],
    new_embedding[:, 1],
):
    ax.text(x_value, y_value, label, fontsize=9)

ax.set_title("LLE transform depends on nearby training examples")
ax.set_xlabel("LLE component 1")
ax.set_ylabel("LLE component 2")
ax.legend(loc="best")
plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Neighborhood Manifolds**</font>


In this lecture, you learned to:
- Explain the manifold assumption through visual examples. 
- Apply Isomap and LLE-family methods to nonlinear datasets. 
- Analyze neighborhood construction, solver choices, and out-of-sample transformation behavior. 

In the next Lecture (Lecture B), we will go over 'Spectral MDS TSNE'